In [2]:
from openai import OpenAI
import pandas as pd
import yaml
import requests
import time
from openpyxl import load_workbook
from openpyxl.styles import PatternFill

# === Konfiguration ===
api_key = "hier API-Key"
base_url = "https://chat-ai.academiccloud.de/v1"
model = "openai-gpt-oss-120b"

# Dateipfade
input_file = "ergebnis_juni_2026_2.xlsx"
output_file = "Ki_juni_2026_2.xlsx"
yaml_file = "relevanz_kriterien.yaml"
esk_yaml_file = "esk_kategorien.yaml"

# === OpenAI-kompatibler Client (SAIA) ===
client = OpenAI(api_key=api_key, base_url=base_url)

# === YAML laden ===
with open(yaml_file, "r", encoding="utf-8") as f:
    kriterien = yaml.safe_load(f)

with open(esk_yaml_file, "r", encoding="utf-8") as f:
    esk_yaml = yaml.safe_load(f)
esk_liste = esk_yaml['esk_kategorien']

# === Excel laden ===
df = pd.read_excel(input_file)
if "Relevanzbewertung" not in df.columns:
    df["Relevanzbewertung"] = ""
if "ESK-Kategorie" not in df.columns:
    df["ESK-Kategorie"] = ""

# === Hilfsfunktionen ===
def lade_ocr_text(url: str) -> str:
    try:
        r = requests.get(url, timeout=10)
        r.raise_for_status()
        text = r.text.replace("\r", " ").replace("\n", " ")
        return text.strip()
    except Exception as e:
        print(f"⚠️ Fehler beim Laden von {url}: {e}")
        return ""

def bewerte_text(ocr_text: str) -> str:
    prompt = f"""
Du bist ein Experte für Buch- und Schriftgeschichte.

Lies den folgenden Text (Inhaltsverzeichnis eines Buches) und beurteile, ob der Inhalt thematisch relevant für das Deutsche Buch- und Schriftmuseum ist.

Antworte ausschließlich im folgenden Format:
Relevanz: relevant / nicht relevant / teilweise relevant
Begründung: [1–2 Sätze]

Relevante Themen:
{', '.join(kriterien['relevante_bereiche'])}

Nicht relevante Themen:
{', '.join(kriterien['nicht_relevant'])}

Beispiele für relevante Titel:
{', '.join(kriterien['positive_beispiele'])}

Beispiele für nicht relevante Titel:
{', '.join(kriterien['negative_beispiele'])}

Text:
---
{ocr_text[:4000]}
---
"""
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1,
        timeout=120
    )
    return response.choices[0].message.content.strip()

def generiere_esk_liste(esk_kategorien):
    codes = []

    for haupt in esk_kategorien:
        codes.append(haupt["code"])
        if "unterkategorien" in haupt:
            for sub in haupt["unterkategorien"]:
                codes.append(sub["code"])

    return codes


esk_liste = generiere_esk_liste(esk_liste)

def kategorisiere_text(ocr_text: str) -> str:
    prompt = f"""
Du bist Experte für Buch-, Schrift- und Medienwissenschaft.

Ordne den folgenden Text der **passendsten ESK-Kategorie(n)** zu, **nur basierend auf den Inhalten, die für uns relevant sind**.
Ignoriere irrelevante Passagen.

**WICHTIGE REGEL:**
- Wähle **immer eine Unterkategorie (z.B. ESK.16.4)**, wenn es inhaltlich möglich ist.
- Nutze eine Hauptkategorie (z.B. ESK.16) **nur dann**, wenn wirklich keine Unterkategorie passt.
- Bevorzuge die **spezifischste passende Kategorie**.

**Hinweis:** Wenn Schlüsselbegriffe im Text vorkommen, nutze die folgende Priorisierung:

ESK.1 – Bibliografische Nachschlagewerke: Nachschlagewerke, Bibliografien, Lexika, Handbücher, Literaturübersichten  
ESK.2 – Buch: Aufbau, Druck, Materialien, Autoren, Buchgeschichte, Rezeption, Leseforschung  
ESK.3 – Beschreibstoffe: Papier, Pergament, Tinte, Papiersorten, Papierherstellung, Wasserzeichen, Blattbildung  
ESK.4 – Restaurierung: Konservierung, Restaurierung, Schadensprävention, Reparaturtechniken  
ESK.5 – Schrift: Schriftarten, Typografie, kalligrafische Techniken, Schriftgestaltung  
ESK.6 – Handschriftenkunde: Handschriften, Schreibtechniken, Handschriftentypen, Analyse von historischen Dokumenten  
ESK.7 – Druck: Drucktechniken, Buchdruck, Offset, Digitaldruck, Druckmaterialien, Druckproduktion  
ESK.8 – Grafische Techniken: Holzschnitt, Kupferstich, Radierung, Federzeichnung, Lithografie, Siebdruck, Illustration, Druckgrafik  
ESK.9 – Fotografie: Fotografie, Bildproduktion, digitale Fotografie, analoge Fotografie, Bildgestaltung  
ESK.10 – Buchgestaltung: Layout, Typografie, Coverdesign, grafische Konzepte, Buchästhetik  
ESK.11 – Buchmalerei: Miniaturen, Initialen, Randdekoration, ornamentale Gestaltung, Malerei  
ESK.12 – Illustration: Illustrationen, Bildkonzepte, Stilrichtungen, Einsatz in Büchern/Medien  
ESK.13 – Einbandkunde: Bucheinbände, Prägung, Vergoldung, Leder, Pappe, Stoff, Schutzfunktion  
ESK.14 – Gebrauchsgrafik: Werbegrafik, Verpackungen, Plakate, Typografie für kommerzielle Zwecke  
ESK.15 – Bibliophilie: Sammeln, Sammlungen, Exlibris, Bücherliebhaberei  
ESK.16 – Zeitungen/Zeitschriften/Presse: Zeitungen, Zeitschriften, Journalismus, Medienproduktion  
ESK.17 – Buchhandel: Verlagswesen, Verlage, Vertrieb, Buchmarkt, Marketing, Buchhandelsorganisation  
ESK.18 – Museumswesen: Museen, Ausstellung, Sammlungspflege, Kuratoren, museale Praxis  
ESK.19 – Bibliothekswesen: Bibliotheken, Benutzerbetreuung, Katalogsysteme, Sammlungspflege  
ESK.20 – Historische Hilfswissenschaften: Chronologie, Heraldik, Genealogie, Siegelkunde, Enzyklopädien, Metrologie, Numismatik, Geschichte, Kulturwissenschaft, Sprachwissenschaft
-Verlage, Verlagswesen und Buchmarkt sind dem Buchhandel (ESK.17) zuzuordnen, auch wenn sie historisch behandelt werden.

Bestimme den Hauptfokus des Textes:

- Liegt der Fokus auf Technik/Praxis → fachliche Kategorie ist Pflicht
- Liegt der Fokus auf historischer Entwicklung → ".4 Geschichte" ist Pflicht
- Bei gemischtem Inhalt → beide Kategorien vergeben

SPEZIALREGEL FÜR GESCHICHTE:

Wenn ein spezifischer Fachbereich (z.B. Bibliotheken, Museen, Buchhandel, Presse) 
UND gleichzeitig historische Entwicklung/Themen im Text vorkommen:

→ Wähle die entsprechende Unterkategorie „Geschichte“ innerhalb dieses Fachbereichs.

Beispiele:
- Bibliotheken + Geschichte → ESK.19.4
- Presse + Geschichte → ESK.16.4
- Buchhandel/Verlage + Geschichte → ESK.17.4
- Grafische Techniken + Geschichte → ESK.8.4

Wähle NICHT ESK.20.Geschichte, wenn ein konkreter Fachbereich erkennbar ist.

Wenn ein Fachbereich erkannt wird UND der Text eindeutig historische Aspekte enthält 
(z.B. Nationalsozialismus, 19. Jahrhundert, Entwicklung, Geschichte, historische Analyse):

→ Wähle IMMER die Unterkategorie ".4 Geschichte" dieses Fachbereichs.

Die Hauptkategorie darf in diesem Fall NICHT verwendet werden.


Wähle ausschließlich aus dieser Liste:
{', '.join(esk_liste)}

Text (nur relevanter Abschnitt):
---
{ocr_text[:4000]}
---

WICHTIG:
- Gib zuerst die passenden ESK-Codes an (kommagetrennt, wenn mehrere zutreffen).
- Danach in einer neuen Zeile eine kurze Begründung (1–2 Sätze).
- Format exakt so:

ESK: [Codes]
Begründung: [kurze Erklärung]

- Keine weiteren Zusätze.
"""
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1,
            timeout=120
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"⚠️ Fehler bei Kategorisierung: {e}")
        return ""


# === Hauptschleife ===
for idx, row in df.iterrows():
    if pd.notna(row["Relevanzbewertung"]) and row["Relevanzbewertung"]:
        continue

    url = row.get("URL")
    if not url or not isinstance(url, str):
        df.at[idx, "Relevanzbewertung"] = "Fehler: keine gültige URL"
        continue

    print(f"🔍 Verarbeite IDN {row['IDN']} …")
    ocr_text = lade_ocr_text(url)
    if not ocr_text:
        df.at[idx, "Relevanzbewertung"] = "Fehler: kein OCR-Text"
        continue

    try:
        # Relevanzbewertung
        bewertung = bewerte_text(ocr_text)
        df.at[idx, "Relevanzbewertung"] = bewertung
        print(f"✅ {row['IDN']} → {bewertung.splitlines()[0]}")

        # ESK-Kategorie nur für relevant / teilweise relevant
        val_lower = bewertung.lower()
        if "relevant" in val_lower and "nicht" not in val_lower:
            esk_kat = kategorisiere_text(ocr_text)
            df.at[idx, "ESK-Kategorie"] = esk_kat
        else:
            df.at[idx, "ESK-Kategorie"] = ""  # explizit leeren

    except Exception as e:
        print(f"❌ Fehler bei IDN {row['IDN']}: {e}")
        df.at[idx, "Relevanzbewertung"] = "Fehler: API"
        df.at[idx, "ESK-Kategorie"] = ""

    time.sleep(2)  # Rate-Limit-Puffer

    
# === NICHT RELEVANTE ZEILEN ENTFERNEN ===
df = df[~df["Relevanzbewertung"].str.lower().str.contains("nicht relevant", na=False)]

# === Excel speichern ===
df.to_excel(output_file, index=False)

# === Farben definieren ===
gruen = PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid")
rot = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")
orange = PatternFill(start_color="FFEB9C", end_color="FFEB9C", fill_type="solid")

# === Excel einfärben ===
wb = load_workbook(output_file)
ws = wb.active

# Spalte Relevanzbewertung finden
col_letter = None
col_idx = None  # Standardmäßig None setzen
for idx, cell in enumerate(ws[1], 1):
    if cell.value == "Relevanzbewertung":
        col_letter = cell.column_letter
        col_idx = idx  # Index der Spalte
        break

if col_letter is None or col_idx is None:
    raise ValueError("Spalte 'Relevanzbewertung' nicht gefunden!")

# Neue Spalte "Übernehmen" einfügen (rechts von "Relevanzbewertung")
ws.insert_cols(col_idx + 1)
ws.cell(row=1, column=col_idx + 1, value="Übernehmen")  # Kopfzeile setzen

# Zellen einfärben
for row in range(2, ws.max_row + 1):
    cell = ws[f"{col_letter}{row}"]
    val = str(cell.value).lower()
    if "relevant" in val and "nicht" not in val and "teilweise" not in val:
        cell.fill = gruen
    elif "nicht relevant" in val:
        cell.fill = rot
    elif "teilweise" in val or "einige punkte" in val:
        cell.fill = orange

# Speichern
wb.save(output_file)
print(f"\n✅ Fertig! Excel mit Farbmarkierungen und ESK-Kategorien gespeichert: {output_file}")


🔍 Verarbeite IDN 1391576810 …
✅ 1391576810 → Relevanz: nicht relevant  
🔍 Verarbeite IDN 1391809378 …
✅ 1391809378 → Relevanz: teilweise relevant  
🔍 Verarbeite IDN 1391809815 …
✅ 1391809815 → Relevanz: relevant  
🔍 Verarbeite IDN 1391812565 …
✅ 1391812565 → Relevanz: relevant  
🔍 Verarbeite IDN 1392044391 …
✅ 1392044391 → Relevanz: nicht relevant  
🔍 Verarbeite IDN 1392530407 …
✅ 1392530407 → Relevanz: nicht relevant  
🔍 Verarbeite IDN 1392736331 …
✅ 1392736331 → Relevanz: nicht relevant  
🔍 Verarbeite IDN 1392742390 …
✅ 1392742390 → Relevanz: relevant  
🔍 Verarbeite IDN 139275013X …
✅ 139275013X → Relevanz: relevant  
🔍 Verarbeite IDN 1392751896 …
✅ 1392751896 → Relevanz: relevant  
🔍 Verarbeite IDN 1393494110 …
✅ 1393494110 → Relevanz: nicht relevant  
🔍 Verarbeite IDN 1393496210 …
✅ 1393496210 → Relevanz: nicht relevant  
🔍 Verarbeite IDN 1393497810 …
✅ 1393497810 → Relevanz: nicht relevant  
🔍 Verarbeite IDN 1394321511 …
✅ 1394321511 → Relevanz: nicht relevant  
🔍 Verarbeite IDN 1